# B2B CRM analysis walkthrough
Synthetic engineered data. Run from this notebook folder or repository root. The completed result tables are already included; this optional notebook is provided unexecuted. See README for the tested rebuild command.

In [ ]:
from pathlib import Path
import pandas as pd
import sqlite3
ROOT = Path.cwd() if (Path.cwd()/'data').exists() else Path.cwd().parent
assert (ROOT/'data/crm.sqlite').exists()
connection = sqlite3.connect(ROOT/'data/crm.sqlite')

## Source conversion and acquisition costs
Check counts before rates. Cost joins must not duplicate source spend.

In [ ]:
source = pd.read_csv(ROOT/'outputs/source_performance.csv')
source

In [ ]:
rates=source.set_index('source').conversion_pct
print('Source rate ratio:',rates['Partner referrals']/rates['Cold outbound'])

## Funnel transitions
These are cumulative reached-stage counts, not current-stage inventory.

In [ ]:
funnel=pd.read_csv(ROOT/'outputs/funnel_by_source.csv')
funnel['proposal_to_win_pct']=100*funnel.won/funnel.proposed
funnel

## Compare raw and source-adjusted rep conversion

In [ ]:
raw=pd.read_csv(ROOT/'outputs/rep_performance.csv')
adjusted=pd.read_csv(ROOT/'outputs/source_adjusted_rep.csv')
raw.merge(adjusted,on='rep_id')

## Try your own SQL
This query uses the generated fully mature cohort. For arbitrary imported data, apply the 90-day cohort CTE from sql/analysis.sql.

In [ ]:
pd.read_sql_query("SELECT industry,COUNT(*) AS leads,SUM(status='Won') AS wins,100.0*SUM(status='Won')/COUNT(*) AS conversion_pct FROM leads GROUP BY industry",connection)

## Interpretation
Why is raw rep rank insufficient? What would establish incremental channel return? Read docs/STAKEHOLDER_MEMO.md before recommending a budget change.

In [ ]:
connection.close()